In [1]:
"""
LoRA 微调前后英翻中性能对比评估脚本
支持 BLEU 和 ROUGE 评分
"""

import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from datasets import Dataset
import numpy as np
from sacrebleu import corpus_bleu
from rouge_score import rouge_scorer
from tqdm import tqdm

# ===========================
# 配置参数
# ===========================
BASE_MODEL_PATH = "/mnt/workspace/.cache/modelscope/models/qwen/Qwen2-1.5B-Instruct"
LORA_MODEL_PATH = "./qwen2-1.5b-en2zh-qlora-test-20k"  # 微调后的 LoRA 权重路径
TEST_DATA_PATH = "./test_data/en2zh_test.jsonl"  # 测试数据集路径
OUTPUT_DIR = "./eval_results"  # 评估结果保存路径
MAX_NEW_TOKENS = 128
BATCH_SIZE = 16  # 根据显存调整

os.makedirs(OUTPUT_DIR, exist_ok=True)

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===========================
# 1. 加载测试数据
# ===========================
def load_test_data(file_path):
    """加载 JSONL 格式的测试数据"""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return Dataset.from_list(data)

In [3]:
# ===========================
# 2. 加载模型和 Tokenizer
# ===========================
def load_base_model(model_path):
    """加载基础模型（未微调）"""
    print(f"🔄 Loading base model from {model_path}...")
    tokenizer = AutoTokenizer.from_pretrained(
        model_path,
        trust_remote_code=True,
        use_fast=False
    )
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"  # 批量推理时使用 left padding

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        trust_remote_code=True,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )
    model.eval()
    return tokenizer, model

def load_lora_model(base_model_path, lora_path):
    """加载微调后的 LoRA 模型"""
    print(f"🔄 Loading LoRA model from {lora_path}...")
    tokenizer = AutoTokenizer.from_pretrained(
        base_model_path,
        trust_remote_code=True,
        use_fast=False
    )
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_path,
        trust_remote_code=True,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )

    model = PeftModel.from_pretrained(base_model, lora_path)
    model.eval()
    return tokenizer, model

In [4]:
# ===========================
# 3. 翻译函数
# ===========================
def translate_batch(model, tokenizer, texts, max_new_tokens=128):
    """批量翻译"""
    prompts = [
        f"<|im_start|>system\nYou are a professional translator. <|im_end|>\n"
        f"<|im_start|>user\nTranslate the following English text to Chinese:\n{text}<|im_end|>\n"
        f"<|im_start|>assistant\n"
        for text in texts
    ]
    
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            top_p=1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        
    # 解码并提取翻译结果
    translations = []
    for i, output in enumerate(outputs):
        decoded = tokenizer.decode(output, skip_special_tokens=False)
        # 提取 assistant 回复部分
        if "<|im_start|>assistant\n" in decoded:
            translation = decoded.split("<|im_start|>assistant\n")[-1]
            translation = translation.split("<|im_end|>")[0].strip()
        else:
            translation = tokenizer.decode(output, skip_special_tokens=True)
        translations.append(translation)
    return translations

In [5]:
# ===========================
# 4. 评估指标计算
# ===========================
def calculate_bleu(predictions, references):
    """计算 BLEU 分数"""
    # sacrebleu 需要 list of references (每个 prediction 对应一个 reference list)
    refs = [[ref] for ref in references]
    bleu = corpus_bleu(predictions, list(zip(*refs)))
    return bleu.score

def calculate_rouge(predictions, references):
    """计算 ROUGE 分数"""
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []
    
    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        rouge1_scores.append(scores['rouge1'].fmeasure)
        rouge2_scores.append(scores['rouge2']. fmeasure)
        rougeL_scores.append(scores['rougeL'].fmeasure)
    
    return {
        'rouge1':  np.mean(rouge1_scores),
        'rouge2': np. mean(rouge2_scores),
        'rougeL': np.mean(rougeL_scores)
    }


In [6]:
# ===========================
# 5. 主评估流程
# ===========================
def evaluate_model(model, tokenizer, test_dataset, model_name, batch_size=8):
    """评估单个模型"""
    print(f"\n📊 Evaluating {model_name}...")
    
    all_predictions = []
    all_references = []
    all_sources = []
    
    # 批量处理
    for i in tqdm(range(0, len(test_dataset), batch_size), desc=f"Translating ({model_name})"):
        batch = test_dataset[i:i+batch_size]
        source_texts = batch['en']
        reference_texts = batch['zh']
        
        predictions = translate_batch(model, tokenizer, source_texts, MAX_NEW_TOKENS)
        
        all_predictions.extend(predictions)
        all_references.extend(reference_texts)
        all_sources.extend(source_texts)
    
    # 计算评估指标
    bleu_score = calculate_bleu(all_predictions, all_references)
    rouge_scores = calculate_rouge(all_predictions, all_references)
    
    results = {
        'model': model_name,
        'bleu': bleu_score,
        'rouge1': rouge_scores['rouge1'],
        'rouge2': rouge_scores['rouge2'],
        'rougeL':  rouge_scores['rougeL'],
        'num_samples': len(all_predictions)
    }
    
    # 保存详细翻译结果
    detailed_results = []
    for src, pred, ref in zip(all_sources, all_predictions, all_references):
        detailed_results.append({
            'source': src,
            'prediction': pred,
            'reference': ref
        })
    
    with open(f"{OUTPUT_DIR}/{model_name}_details.jsonl", 'w', encoding='utf-8') as f:
        for item in detailed_results:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    
    return results

In [7]:
# ===========================
# 6. 主函数
# ===========================

print("=" * 60)
print("🚀 LoRA 微调前后英翻中性能对比评估")
print("=" * 60)
    
# 加载测试数据
test_dataset = load_test_data(TEST_DATA_PATH)
print(f"✅ Loaded {len(test_dataset)} test samples")
    
# 评估基础模型
tokenizer_base, model_base = load_base_model(BASE_MODEL_PATH)
results_base = evaluate_model(model_base, tokenizer_base, test_dataset, "base_model", BATCH_SIZE)
    
# 清理显存
del model_base
torch.cuda.empty_cache()
    


🚀 LoRA 微调前后英翻中性能对比评估
✅ Loaded 1000 test samples
🔄 Loading base model from /mnt/workspace/.cache/modelscope/models/qwen/Qwen2-1.5B-Instruct...


`torch_dtype` is deprecated! Use `dtype` instead!



📊 Evaluating base_model...


Translating (base_model): 100%|██████████| 63/63 [01:17<00:00,  1.23s/it]


In [8]:
# 评估 LoRA 微调模型
tokenizer_lora, model_lora = load_lora_model(BASE_MODEL_PATH, LORA_MODEL_PATH)
results_lora = evaluate_model(model_lora, tokenizer_lora, test_dataset, "lora_model", BATCH_SIZE)

🔄 Loading LoRA model from ./qwen2-1.5b-en2zh-qlora-test-20k...

📊 Evaluating lora_model...


Translating (lora_model): 100%|██████████| 63/63 [02:19<00:00,  2.22s/it]


In [9]:
# 汇总结果
print("\n" + "=" * 60)
print("📊 评估结果对比")
print("=" * 60)
print(f"\n{'Metric':<15} {'Base Model':<15} {'LoRA Model':<15} {'Improvement':<15}")
print("-" * 60)
print(f"{'BLEU':<15} {results_base['bleu']:<15.2f} {results_lora['bleu']:<15.2f} {results_lora['bleu'] - results_base['bleu']:<15.2f}")
print(f"{'ROUGE-1':<15} {results_base['rouge1']:<15.4f} {results_lora['rouge1']:<15.4f} {results_lora['rouge1'] - results_base['rouge1']:<15.4f}")
print(f"{'ROUGE-2':<15} {results_base['rouge2']: <15.4f} {results_lora['rouge2']:<15.4f} {results_lora['rouge2'] - results_base['rouge2']:<15.4f}")
print(f"{'ROUGE-L': <15} {results_base['rougeL']:<15.4f} {results_lora['rougeL']:<15.4f} {results_lora['rougeL'] - results_base['rougeL']:<15.4f}")
    
# 保存汇总结果
summary = {
        'base_model':  results_base,
        'lora_model': results_lora,
        'improvement': {
            'bleu': results_lora['bleu'] - results_base['bleu'],
            'rouge1': results_lora['rouge1'] - results_base['rouge1'],
            'rouge2': results_lora['rouge2'] - results_base['rouge2'],
            'rougeL': results_lora['rougeL'] - results_base['rougeL']
    }
}

from datetime import datetime

timestamp = datetime.now().strftime("%m%d_%H%M")
OUTPUT_DIR_WITH_TIMESTAMP = os.path.join(OUTPUT_DIR, f"{timestamp}")
os.makedirs(OUTPUT_DIR_WITH_TIMESTAMP, exist_ok=True)
    
with open(f"{OUTPUT_DIR_WITH_TIMESTAMP}/summary.json", 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
    
print(f"\n✅ 结果已保存到 {OUTPUT_DIR_WITH_TIMESTAMP}/")


📊 评估结果对比

Metric          Base Model      LoRA Model      Improvement    
------------------------------------------------------------
BLEU            6.30            17.51           11.21          
ROUGE-1         0.4002          0.4337          0.0335         
ROUGE-2         0.2130          0.2480          0.0349         
ROUGE-L         0.3958          0.4312          0.0354         

✅ 结果已保存到 ./eval_results/1216_2036/
